In [0]:
import mlflow
from pyspark.sql.functions import struct, col

# 1. Define the full Unity Catalog model name (catalog.schema.model)
model_name = "workspace.default.nvda_price_classifier"

# 2. Check if model has any versions registered
client = mlflow.tracking.MlflowClient()
versions = client.search_model_versions(f"name='{model_name}'")

if len(versions) == 0:
    raise ValueError(
        f"No versions found for model '{model_name}'. "
        "You need to train and log a model version first. "
        "Run your training notebook to register a model version."
    )

# Get the latest version (versions are sorted by creation time, descending)
latest_version = versions[0].version
print(f"Loading model version: {latest_version}")

# 3. Load the model using Unity Catalog syntax with version number
model_uri = f"models:/{model_name}/{latest_version}"
predict_fn = mlflow.pyfunc.spark_udf(spark, model_uri)

# 4. Read data Gold
df_gold = spark.table("gold_nvda_features")

# 5. Execute prediction
features = ["open_price", "high_price", "low_price", "close_price", "volume", "sma_7", "sma_30", "daily_volatility"]
df_predictions = df_gold.withColumn(
    "prediction", 
    predict_fn(struct([col(c) for c in features]))
)

display(df_predictions.select("date", "close_price", "prediction").orderBy(col("date").desc()).limit(10))

In [0]:

df_predictions.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("predictions_nvda")